# Kiểm tra GPU của runtime Colab

Cách kết nối trong VS Code: **Select Kernel** (góc trên bên phải) → **Colab** → **New Colab Server** → chọn loại GPU (T4 / L4 / A100) → chạy lần lượt từng cell.

Trên trình duyệt Colab: **Runtime → Change runtime type → GPU**.

In [1]:
# 1. Thông tin runtime: có đang chạy trên Colab không, hệ điều hành, Python
import sys, platform, os, socket

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Đang chạy trên Colab:", IN_COLAB)
print("Hostname           :", socket.gethostname())
print("Hệ điều hành       :", platform.platform())
print("Python             :", sys.version.split()[0])
print("Số CPU             :", os.cpu_count())
try:
    import psutil
    print(f"RAM                : {psutil.virtual_memory().total / 1024**3:.1f} GB")
except ImportError:
    pass

Đang chạy trên Colab: True
Hostname           : 0915733c42a8
Hệ điều hành       : Linux-6.6.122+-x86_64-with-glibc2.35
Python             : 3.12.13
Số CPU             : 2
RAM                : 12.7 GB


In [2]:
# 2. nvidia-smi: danh sách GPU, driver, CUDA, bộ nhớ
import shutil, subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,driver_version,compute_cap",
         "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("KHÔNG tìm thấy nvidia-smi -> runtime này không có GPU NVIDIA.")

Sat Sep 26 08:36:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 3. PyTorch nhìn thấy GPU nào
import torch

print("PyTorch        :", torch.__version__)
print("CUDA (build)   :", torch.version.cuda)
print("cuDNN          :", torch.backends.cudnn.version())
print("CUDA available :", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | {p.total_memory / 1024**3:.1f} GB | compute capability {p.major}.{p.minor} "
          f"| {p.multi_processor_count} SM")

PyTorch        : 2.11.0+cu128
CUDA (build)   : 12.8
cuDNN          : 91900
CUDA available : True
  GPU 0: Tesla T4 | 14.6 GB | compute capability 7.5 | 40 SM


In [4]:
# 4. TensorFlow nhìn thấy GPU nào (bật memory growth để không chiếm hết VRAM)
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow   :", tf.__version__)
print("Built w/ CUDA:", tf.test.is_built_with_cuda())
print("GPU          :", gpus if gpus else "không có")
for g in gpus:
    print("  ", tf.config.experimental.get_device_details(g))

TensorFlow   : 2.20.0
Built w/ CUDA: True
GPU          : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
   {'compute_capability': (7, 5), 'device_name': 'Tesla T4'}


In [5]:
# 5. Benchmark nhanh: nhân ma trận trên GPU vs CPU (PyTorch)
import time

def bench(device, n=4096, reps=10, dtype=torch.float32):
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    (a @ b)  # warm-up
    if device == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    dt = (time.perf_counter() - t) / reps
    return dt, 2 * n**3 / dt / 1e12

dt, tflops = bench("cpu", n=2048, reps=3)
print(f"CPU  fp32 (2048): {dt * 1e3:8.1f} ms/lần | {tflops:6.2f} TFLOPS")
if torch.cuda.is_available():
    for dtype in [torch.float32, torch.float16]:
        dt, tflops = bench("cuda", dtype=dtype)
        print(f"GPU  {str(dtype).split('.')[-1]:7s} (4096): {dt * 1e3:8.2f} ms/lần | {tflops:6.2f} TFLOPS")
    print(f"\nVRAM đã cấp phát: {torch.cuda.memory_allocated() / 1024**2:.0f} MB | "
          f"đã giữ: {torch.cuda.memory_reserved() / 1024**2:.0f} MB")
else:
    print("Không có GPU CUDA -> bỏ qua benchmark GPU.")

CPU  fp32 (2048):    128.1 ms/lần |   0.13 TFLOPS
GPU  float32 (4096):    38.52 ms/lần |   3.57 TFLOPS
GPU  float16 (4096):     3.02 ms/lần |  45.53 TFLOPS

VRAM đã cấp phát: 8 MB | đã giữ: 212 MB
